# Week 5 — M2 Capstone Stop 2: Optimizing the RAG Pipeline

This notebook covers **Stop 2** of the M2 Capstone RAG Knowledge Base project.
It builds on the basic end-to-end RAG pipeline from Stop 1 (Week 4) by adding:

| Component | What | Why |
|---|---|---|
| **1 — MMR Retrieval** | Replace similarity search with Maximal Marginal Relevance | Reduces redundant chunks from the same section |
| **2 — Cross-Encoder Re-Ranking** | Re-order MMR results using a transformer cross-encoder | Precision-ranks by true query-document relevance |
| **3 — Chunk Size Experiment** | Compare 250 / 500 / 1000-char chunk sizes | Empirically validates the Stop 1 default |
| **4 — Retrieval Quality Metrics** | Precision@k and MRR, baseline vs. optimised | Quantifies the improvement |

**Prerequisite:** Stop 1 pipeline must be working (chroma_db built).  
**Source code:** `c03-t05-bruno-pieri-m2-challenge/src/pipeline/`

## Setup

In [ ]:
import os
import sys
from pathlib import Path

# ── Anchor paths relative to this notebook's location ──
NOTEBOOK_DIR = Path(os.getcwd())          # project root (techstore-chatbot/)
CAPSTONE_ROOT = NOTEBOOK_DIR / "c03-t05-bruno-pieri-m2-challenge"

# Change CWD so that data/ and chroma_db/ resolve correctly inside src/
os.chdir(CAPSTONE_ROOT)
if str(CAPSTONE_ROOT) not in sys.path:
    sys.path.insert(0, str(CAPSTONE_ROOT))

from dotenv import load_dotenv
load_dotenv()   # picks up .env from capstone root or parent dirs

print(f"CWD          : {os.getcwd()}")
print(f"OPENAI key   : {'set ✓' if os.environ.get('OPENAI_API_KEY') else 'MISSING ✗'}")

---
## Component 1 — MMR Retrieval

**Why MMR instead of plain similarity search?**

Standard cosine-similarity retrieval ranks candidates by proximity to the query vector. When a corpus has several near-identical chunks (e.g. the warranty clause appears verbatim in both `policy_warranty_terms.txt` and each product manual), all top-k slots may be occupied by chunks from the *same* document — wasting the LLM's context window with redundant text.

**Maximal Marginal Relevance** selects the first document by pure similarity, then each subsequent document maximises:

```
λ · sim(d, query) − (1−λ) · max_sim(d, already_selected)
```

With `λ=0.85` (favoring relevance while preserving diversity) and `fetch_k=20 >> k=6`, MMR has enough candidates to find results that are both *relevant* and *diverse*.

In [ ]:
from src.pipeline.vectorstore import load_vectorstore, build_vectorstore
from src.pipeline.loader import load_documents, chunk_documents

# Load existing vectorstore (build on first run if chroma_db is missing)
chroma_path = Path("chroma_db")
if chroma_path.exists() and any(chroma_path.iterdir()):
    vs = load_vectorstore()
else:
    print("chroma_db not found — building from corpus (calls OpenAI embeddings API)...")
    docs = load_documents()
    chunks = chunk_documents(docs)
    vs = build_vectorstore(chunks)

print(f"\nVectorstore ready — {vs._collection.count()} embedded chunks")

In [ ]:
from src.pipeline.vectorstore import get_mmr_retriever

retriever = get_mmr_retriever(vs)

# A broad query that touches multiple corpus sections.
# With similarity search, all results would cluster around one topic;
# with MMR they should span different documents.
broad_query = "TechStore Plus products support warranty return policy"

mmr_docs = retriever.invoke(broad_query)

print(f"MMR returned {len(mmr_docs)} documents for broad query")
print("\nSources (expect chunks from multiple distinct files):")
seen = set()
for i, doc in enumerate(mmr_docs):
    src = doc.metadata["source"]
    marker = " ← NEW" if src not in seen else ""
    seen.add(src)
    print(f"  [{i+1}] {src}{marker}")

unique_sources = len({d.metadata['source'] for d in mmr_docs})
print(f"\nUnique source files: {unique_sources} / {len(mmr_docs)}")
print("✓ Diversity check passed" if unique_sources >= 3 else "⚠ Low diversity — check fetch_k")

---
## Component 2 — Cross-Encoder Re-Ranking

**Why a cross-encoder after MMR?**

The MMR retriever uses a *bi-encoder*: query and document are embedded **separately** and compared in a shared vector space. This is fast (O(1) per query after indexing) but lossy — the fixed-size embeddings cannot model fine-grained interactions between specific query tokens and document tokens.

A *cross-encoder* (`cross-encoder/ms-marco-MiniLM-L-6-v2`) processes the `(query, document)` pair **jointly** through a transformer, capturing cross-attention between every query token and every document token. This is much more accurate at judging relevance, at the cost of O(n) forward passes per candidate set.

**Two-stage pipeline:** MMR (fast, fetch_k=20 → k=6) pre-filters, then the cross-encoder precision-sorts the survivors. Only the top-3 reach the LLM.

In [ ]:
from src.pipeline.reranker import rerank, RERANK_TOP_N

rerank_query = "differences between standard and extended warranty coverage"

# Stage 1 — MMR retrieval
mmr_candidates = retriever.invoke(rerank_query)
print(f"MMR stage: {len(mmr_candidates)} candidate chunks")
print("Before re-ranking:")
for i, doc in enumerate(mmr_candidates):
    print(f"  [{i+1}] {doc.metadata['source']}")

print()

# Stage 2 — Cross-encoder re-ranking
top_docs = rerank(rerank_query, mmr_candidates)
print(f"Re-rank stage: top {RERANK_TOP_N} passed to LLM")
print("After re-ranking (with relevance scores):")
for i, doc in enumerate(top_docs):
    score = doc.metadata.get("rerank_score", 0.0)
    print(f"  [{i+1}] score={score:+.4f} | {doc.metadata['source']}")
    print(f"        {doc.page_content[:120].strip()}...")

print(f"\n✓ Context reduced from {len(mmr_candidates)} → {len(top_docs)} chunks")

---
## Component 3 — Chunk Size Experiment

Three configurations were tested against the same 5 Stop 2 queries.
Full analysis: `c03-t05-bruno-pieri-m2-challenge/docs/chunk-experiment.md`

In [ ]:
chunk_experiment = [
    {
        "config": "chunk_size=250, overlap=25",
        "avg_chunks": 4,
        "relevance": 3,
        "quality": 3,
        "notes": "Fragmentation — warranty clause split mid-sentence; LLM reconstructs partial rules",
    },
    {
        "config": "chunk_size=500, overlap=50  ← baseline",
        "avg_chunks": 4,
        "relevance": 5,
        "quality": 5,
        "notes": "Best balance: one policy section per chunk, minimal redundancy",
    },
    {
        "config": "chunk_size=1000, overlap=100",
        "avg_chunks": 4,
        "relevance": 4,
        "quality": 4,
        "notes": "One long chunk dominates retrieval; context diversity drops",
    },
]

header = f"{'Configuration':<40} {'Chunks':>6} {'Rel(1-5)':>8} {'Qual(1-5)':>9}  Notes"
print(header)
print("-" * len(header))
for row in chunk_experiment:
    print(f"{row['config']:<40} {row['avg_chunks']:>6} {row['relevance']:>8} {row['quality']:>9}  {row['notes']}")

print("\nConclusion: chunk_size=500, overlap=50 — highest scores on both relevance and quality.")

---
## Component 4 — Retrieval Quality Metrics

We compare:
- **Baseline** — Stop 1 similarity retriever, k=4
- **Optimised** — MMR (fetch_k=20, k=6) → cross-encoder top-3

Metrics are computed across 10 questions with known relevant documents.
Implementation: `src/pipeline/metrics.py`

In [ ]:
from src.pipeline.metrics import precision_at_k, mrr, evaluate_retriever, EVAL_SET

# Quick unit checks
p3 = precision_at_k(["a", "b", "c"], ["a", "c"], k=3)
p1 = precision_at_k(["a", "b", "c"], ["a", "c"], k=1)
m  = mrr(["b", "a", "c"], ["a"])

assert abs(p3 - 2/3) < 1e-9, f"Expected ~0.6667, got {p3}"
assert p1 == 1.0,             f"Expected 1.0, got {p1}"
assert m  == 0.5,             f"Expected 0.5, got {m}"

print(f"precision_at_k([a,b,c], relevant=[a,c], k=3) = {p3:.4f}  (expected 0.6667)")
print(f"precision_at_k([a,b,c], relevant=[a,c], k=1) = {p1:.4f}  (expected 1.0000)")
print(f"mrr([b,a,c], relevant=[a])                   = {m:.4f}  (expected 0.5000)")
print("\n✓ Metric functions verified")
print(f"Evaluation set: {len(EVAL_SET)} queries")

In [ ]:
# ── Baseline retriever (Stop 1 — simple cosine similarity, k=4) ──
baseline_retriever = vs.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

print("Running baseline evaluation (similarity k=4) ...")
baseline_results, baseline_agg = evaluate_retriever(
    retriever_fn=lambda q: baseline_retriever.invoke(q),
    eval_set=EVAL_SET,
)

print("Done.")
print(f"  Precision@3 : {baseline_agg['precision@3']:.2f}")
print(f"  Precision@6 : {baseline_agg['precision@6']:.2f}")
print(f"  MRR         : {baseline_agg['mrr']:.2f}")

In [ ]:
# ── Optimised pipeline (MMR k=6 → cross-encoder top-3) ──
def optimised_retriever_fn(query: str):
    """MMR retrieval followed by cross-encoder re-ranking."""
    mmr_docs = retriever.invoke(query)       # 6 diverse candidates
    top_docs = rerank(query, mmr_docs)       # cross-encoder → top-3
    return top_docs

print("Running optimised evaluation (MMR k=6 + rerank top-3) ...")
optimised_results, optimised_agg = evaluate_retriever(
    retriever_fn=optimised_retriever_fn,
    eval_set=EVAL_SET,
)

print("Done.")
print(f"  Precision@3 : {optimised_agg['precision@3']:.2f}")
print(f"  Precision@6 : {optimised_agg['precision@6']:.2f}")
print(f"  MRR         : {optimised_agg['mrr']:.2f}")

In [ ]:
# ── Comparison table ──
def delta(opt, base):
    d = opt - base
    return f"+{d:.2f}" if d >= 0 else f"{d:.2f}"

print(f"{'Pipeline':<35} {'P@3':>6} {'P@6':>6} {'MRR':>6}")
print("-" * 60)
print(f"{'Baseline (similarity, k=4)':<35} "
      f"{baseline_agg['precision@3']:>6.2f} "
      f"{baseline_agg['precision@6']:>6.2f} "
      f"{baseline_agg['mrr']:>6.2f}")
print(f"{'Optimised (MMR k=6 + rerank top-3)':<35} "
      f"{optimised_agg['precision@3']:>6.2f} "
      f"{optimised_agg['precision@6']:>6.2f} "
      f"{optimised_agg['mrr']:>6.2f}")
print("-" * 60)
print(f"{'Delta':<35} "
      f"{delta(optimised_agg['precision@3'], baseline_agg['precision@3']):>6} "
      f"{delta(optimised_agg['precision@6'], baseline_agg['precision@6']):>6} "
      f"{delta(optimised_agg['mrr'], baseline_agg['mrr']):>6}")

mrr_ok = optimised_agg['mrr'] >= baseline_agg['mrr']
print(f"\nStop 2 requirement (optimised MRR ≥ baseline MRR): {'✓ PASSED' if mrr_ok else '✗ FAILED'}")

In [ ]:
# ── Per-query breakdown ──
print(f"{'#':<3} {'Query':<52} {'P@3 base':>8} {'P@3 opt':>7} {'MRR base':>8} {'MRR opt':>7}")
print("-" * 92)
for i, (br, or_) in enumerate(zip(baseline_results, optimised_results), 1):
    q = br.query[:50] + ".." if len(br.query) > 50 else br.query
    print(f"{i:<3} {q:<52} "
          f"{br.precision_at_3:>8.2f} "
          f"{or_.precision_at_3:>7.2f} "
          f"{br.mrr_score:>8.2f} "
          f"{or_.mrr_score:>7.2f}")

---
## Five Stop 2 Queries — End-to-End Pipeline

Each query runs through the full optimised pipeline:
1. MMR retrieval (fetch_k=20 → k=6 diverse chunks)
2. Cross-encoder re-ranking (→ top-3)
3. LLM answer generation with source citations

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a TechStore Plus customer support assistant. "
     "Answer the question using ONLY the provided context. "
     "End every factual claim with the source filename in brackets, e.g. [policy_return_policy.txt]."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

chain = rag_prompt | llm


def run_stop2_query(question: str) -> str:
    """Full optimised pipeline: MMR → rerank → LLM."""
    mmr_candidates = retriever.invoke(question)
    top_docs = rerank(question, mmr_candidates)
    context = "\n\n".join(
        f"[{doc.metadata['source']}]\n{doc.page_content}"
        for doc in top_docs
    )
    response = chain.invoke({"context": context, "question": question})
    return response.content


print("RAG chain ready — using gpt-4.1-mini with optimised retriever")

In [ ]:
stop2_queries = [
    "What is TechStore Plus's return policy?",
    "How do I troubleshoot a router that won't connect to the internet?",
    "What does the Premium Protection Plan cover?",
    "How much RAM does the Laptop Pro X1 have?",
    "What are the steps to file a warranty claim?",
]

for i, query in enumerate(stop2_queries, 1):
    print(f"\n{'='*70}")
    print(f"Query {i}: {query}")
    print("=" * 70)
    answer = run_stop2_query(query)
    print(answer)

---
## Stop 2 Checklist

| Item | Status |
|---|---|
| MMR retriever active (fetch_k=20, k=6) | ✅ `src/pipeline/vectorstore.get_mmr_retriever()` |
| Diversity verified for broad query | ✅ Cell above shows ≥3 unique source files |
| Cross-encoder re-ranker → top-3 | ✅ `src/pipeline/reranker.rerank()` |
| Chunk size experiment (≥2 configs documented) | ✅ `docs/chunk-experiment.md` — 250/500/1000 tested |
| `precision_at_k` and `mrr` as Python functions | ✅ `src/pipeline/metrics.py` |
| Precision@6 and MRR computed for both pipelines | ✅ Comparison table above |
| Optimised MRR ≥ baseline MRR | ✅ Verified programmatically |
| All 5 Stop 2 queries run end-to-end without errors | ✅ Cell above |

---

**Next:** Stop 3 (Week 6) adds Graph RAG, guardrails (claim verification), and multimodal table retrieval to create the production `TechStoreRAGAgent`.